# Geomar AI Oxygen - Hypoxia Prediction Training (Google Colab)

This notebook runs the weighted hypoxia prediction model training pipeline on Google Colab.

**Project**: Boknis Eck hypoxia prediction using weighted Temporal Fusion Transformer  
**Repository**: https://github.com/YOUR_USERNAME/Geomar_AI_Oxygen

## Notebook Overview

1. **Setup**: Install dependencies, clone repository, mount Google Drive for checkpoints
2. **Data Preparation**: Run data ingestion pipeline (Phases 2-5)
3. **Training Options**:
   - Quick test training (5 epochs)
   - Full training with default hyperparameters
   - Hyperparameter tuning with Optuna
   - Weighted loss verification test
4. **Results**: View training metrics, download checkpoints

## Requirements

- GPU runtime recommended (Runtime > Change runtime type > GPU)
- Google Drive for saving checkpoints (optional but recommended)


# 1. Setup Environment

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected. Training will be slower on CPU.")
    print("   Go to Runtime > Change runtime type > Hardware accelerator > GPU")

In [ ]:
# Clone the repository (update with your GitHub URL)
import os

REPO_URL = "https://github.com/YOUR_USERNAME/Geomar_AI_Oxygen.git"  # UPDATE THIS
REPO_NAME = "Geomar_AI_Oxygen"

if os.path.exists(REPO_NAME):
    print(f"Repository already exists. Pulling latest changes...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL}

# Change to repository directory
os.chdir(REPO_NAME)
print(f"\nCurrent directory: {os.getcwd()}")
print(f"Repository contents:")
!ls -la

In [ ]:
# Install dependencies
print("Installing project dependencies...\n")
!pip install -q -r requirements.txt

# Install optuna for hyperparameter tuning
!pip install -q optuna

print("\n✓ Dependencies installed successfully!")

# Verify key imports
import pytorch_forecasting
import lightning.pytorch as pl
import optuna
import pandas as pd
import numpy as np

print(f"\nKey package versions:")
print(f"  pytorch-forecasting: {pytorch_forecasting.__version__}")
print(f"  lightning: {pl.__version__}")
print(f"  optuna: {optuna.__version__}")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")

In [ ]:
# Mount Google Drive for saving checkpoints (OPTIONAL)
# This allows you to keep trained models even after the Colab session ends

from google.colab import drive
import os

MOUNT_DRIVE = True  # Set to False to skip Drive mounting

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    
    # Create checkpoint directory in Drive
    CHECKPOINT_DIR = '/content/drive/MyDrive/Geomar_Checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"\n✓ Checkpoints will be saved to: {CHECKPOINT_DIR}")
else:
    # Use local Colab storage (will be lost when session ends)
    CHECKPOINT_DIR = 'models/hypoxia_tft'
    print(f"\n⚠️  Checkpoints will be saved locally (session only): {CHECKPOINT_DIR}")

# 2. Verify Data and Pipeline

In [ ]:
# Check data files
print("Data files in Documentation/data/:")
!ls -lh Documentation/data/

print("\nVerifying data ingestion...")
from src import data_ingestion

# Test loading ocean data
df_ocean = data_ingestion._load_ocean_data()
print(f"✓ Ocean data loaded: {len(df_ocean)} rows")
print(f"  Date range: {df_ocean['Date'].min()} to {df_ocean['Date'].max()}")
print(f"  Depths: {sorted(df_ocean['Depth_m'].unique())}")

In [ ]:
# Run unit tests to verify pipeline integrity
print("Running unit tests...\n")
!python -m pytest tests/ -v --tb=short

print("\n✓ All tests passed! Pipeline is ready.")

# 3. Training Options

Choose one of the following training modes:
- **Quick Test**: 5 epochs, small batch size, for testing
- **Full Training**: Default hyperparameters, full epochs
- **Hyperparameter Tuning**: Optuna search for best hyperparameters
- **Weighted Loss Verification**: Test that weighting mechanism works

## Option A: Quick Test Training (5 epochs)

In [ ]:
# Quick test training - 5 epochs, small batch
print("Starting quick test training (5 epochs)...\n")

!python train.py \
    --max-epochs 5 \
    --batch-size 32 \
    --checkpoint-path {CHECKPOINT_DIR} \
    --patience 2

print("\n✓ Quick test complete!")
print(f"Check results in: {CHECKPOINT_DIR}")

## Option B: Full Training (Default Hyperparameters)

In [ ]:
# Full training with default hyperparameters from SPEC.md
print("Starting full training with default hyperparameters...\n")

!python train.py \
    --max-epochs 100 \
    --batch-size 64 \
    --checkpoint-path {CHECKPOINT_DIR} \
    --patience 3 \
    --learning-rate 0.03 \
    --hidden-size 16 \
    --attention-head-size 1 \
    --dropout 0.1

print("\n✓ Full training complete!")
print(f"Check results in: {CHECKPOINT_DIR}")

## Option C: Hyperparameter Tuning with Optuna

In [ ]:
# Hyperparameter tuning with Optuna (WARNING: This takes a long time!)
N_TRIALS = 20  # Reduce for faster tuning (50 recommended for production)

print(f"Starting hyperparameter tuning with {N_TRIALS} trials...\n")
print("⏱️  This will take several hours. Monitor progress below.\n")

!python tune_hyperparameters.py \
    --n-trials {N_TRIALS} \
    --output tuned_hyperparameters.json \
    --study-name colab_tuning

print("\n✓ Hyperparameter tuning complete!")
print("Best hyperparameters saved to: tuned_hyperparameters.json")

# Display best hyperparameters
import json
with open('tuned_hyperparameters.json', 'r') as f:
    tuned = json.load(f)

print("\nBest hyperparameters:")
for key, value in tuned['hyperparameters'].items():
    print(f"  {key}: {value}")
print(f"\nBest validation loss: {tuned['val_loss']:.4f}")

In [ ]:
# Train with tuned hyperparameters (run after tuning completes)
print("Training with tuned hyperparameters...\n")

!python train.py \
    --load-hyperparameters tuned_hyperparameters.json \
    --checkpoint-path {CHECKPOINT_DIR} \
    --max-epochs 100 \
    --patience 3

print("\n✓ Training with tuned hyperparameters complete!")

## Option D: Weighted Loss Verification Test

In [ ]:
# Verify that weighted loss mechanism is working
print("Running weighted loss verification test...\n")
print("This trains two models: extreme weighted (100x) vs uniform (1x)\n")

!python verify_weighted_loss.py

print("\n✓ Verification complete! Check output above for pass/fail.")

# 4. View Results and Download Checkpoints

In [ ]:
# View training metadata
import json
from pathlib import Path

metadata_path = Path(CHECKPOINT_DIR) / "training_metadata.json"

if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    print("="*80)
    print("TRAINING METADATA")
    print("="*80)
    
    print(f"\nTraining Date: {metadata['training_date']}")
    
    print("\nHyperparameters:")
    for key, value in metadata['hyperparameters'].items():
        print(f"  {key}: {value}")
    
    print("\nDataset Info:")
    for key, value in metadata['dataset_info'].items():
        if isinstance(value, dict):
            print(f"  {key}:")
            for k, v in value.items():
                print(f"    {k}: {v}")
        else:
            print(f"  {key}: {value}")
    
    print("\nFeatures Used:")
    for feature in metadata['features']:
        print(f"  - {feature}")
    
    print("\nWeight Configuration:")
    print(f"  Thresholds: {metadata['weight_configuration']['thresholds']}")
    print(f"  Tier Weights: {metadata['weight_configuration']['tier_weights']}")
else:
    print(f"⚠️  No training metadata found at {metadata_path}")
    print("   Run training first (Option A, B, or C above)")

In [ ]:
# List all checkpoint files
print(f"Checkpoint files in {CHECKPOINT_DIR}:\n")
!ls -lh {CHECKPOINT_DIR}

# Check best model checkpoint
best_ckpt = Path(CHECKPOINT_DIR) / "best_model.ckpt"
if best_ckpt.exists():
    size_mb = best_ckpt.stat().st_size / 1e6
    print(f"\n✓ Best model checkpoint found: {best_ckpt}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print("\n⚠️  No best model checkpoint found. Run training first.")

In [ ]:
# Download checkpoint to local machine (if not using Google Drive)
if not MOUNT_DRIVE:
    from google.colab import files
    
    print("Downloading checkpoint files...\n")
    
    # Download best model checkpoint
    best_ckpt = Path(CHECKPOINT_DIR) / "best_model.ckpt"
    if best_ckpt.exists():
        files.download(str(best_ckpt))
        print(f"✓ Downloaded: best_model.ckpt")
    
    # Download metadata
    metadata_path = Path(CHECKPOINT_DIR) / "training_metadata.json"
    if metadata_path.exists():
        files.download(str(metadata_path))
        print(f"✓ Downloaded: training_metadata.json")
    
    # Download last checkpoint (if exists)
    last_ckpt = Path(CHECKPOINT_DIR) / "last.ckpt"
    if last_ckpt.exists():
        files.download(str(last_ckpt))
        print(f"✓ Downloaded: last.ckpt")
else:
    print(f"Checkpoints are already saved to Google Drive: {CHECKPOINT_DIR}")
    print("You can access them from your Drive even after this session ends.")

# 5. Cleanup (Optional)

In [ ]:
# Clean up temporary files to free disk space
# WARNING: This will delete local checkpoints if you didn't mount Google Drive!

import shutil

CLEANUP = False  # Set to True to enable cleanup

if CLEANUP:
    if not MOUNT_DRIVE:
        print("⚠️  WARNING: You haven't mounted Google Drive!")
        print("   Checkpoints will be DELETED. Download them first!")
        print("   Set CLEANUP = False to cancel.")
    
    # Remove cache files
    if Path('.weather_cache').exists():
        shutil.rmtree('.weather_cache')
        print("✓ Removed weather cache")
    
    if Path('__pycache__').exists():
        shutil.rmtree('__pycache__')
        print("✓ Removed Python cache")
    
    print("\nCleanup complete!")
else:
    print("Cleanup disabled. Set CLEANUP = True to enable.")

# Notes

## Training Time Estimates (on T4 GPU)
- **Quick test (5 epochs)**: ~5-10 minutes
- **Full training (100 epochs)**: ~1-2 hours (with early stopping, typically stops around 20-40 epochs)
- **Hyperparameter tuning (20 trials)**: ~4-8 hours
- **Hyperparameter tuning (50 trials)**: ~10-20 hours

## Tips
1. **Use GPU runtime** for 10-50x speedup (Runtime > Change runtime type > GPU)
2. **Mount Google Drive** to keep checkpoints after session ends
3. **Start with quick test** to verify everything works before long training runs
4. **Download checkpoints** if not using Drive (they'll be lost when session ends)
5. **Monitor GPU usage** with `!nvidia-smi` to ensure GPU is being used

## Repository Structure
```
Geomar_AI_Oxygen/
├── src/
│   ├── data_ingestion.py   # Phase 2: Load ocean + weather data
│   ├── pipeline.py         # Phase 3: Weekly resampling
│   ├── labeling.py         # Phase 4: Hypoxia labeling + weights
│   ├── features.py         # Phase 5: Feature engineering
│   ├── dataset.py          # Phase 6: Dataset construction
│   └── model.py            # Phase 7: TFT model definition
├── train.py                # Phase 8: Full training pipeline
├── tune_hyperparameters.py # Optuna hyperparameter search
├── verify_weighted_loss.py # Weighted loss verification test
└── Documentation/
    ├── SPEC.md             # Technical specification
    └── BUILD_PLAN.md       # Implementation phases
```

## Next Steps After Training
1. Implement Phase 9: Evaluation suite (`evaluate.py`)
2. Implement Phase 10: Streamlit dashboard (`app.py`)
3. Run evaluation on held-out hypoxic episodes
4. Tune hyperparameters if default training shows poor tail metrics
